# Oracle AI Agent Memory — Developer Guide (OCI)

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

A hands-on, step-by-step guide to the `oracleagentmemory` AI Agent Package for AI developers. This OCI variant teaches the API surface while using OCI Generative AI for embeddings, automatic extraction, summaries, and chat examples.

**What you'll learn**
1. How to connect the package to an Oracle AI Database instance
2. The three core primitives: **users/agents**, **memories**, and **threads**
3. Manual vs. **automatic** LLM-powered memory extraction with OCI Generative AI
4. How to retrieve relevant context with **vector search** and **context cards**
5. How to **scope** searches across multiple users and agents
6. How to plug the package into an LLM agent loop
7. How to **persist and resume** a conversation across sessions
8. How to build agents that adapt to and learn from new information over time

**Prerequisites**
- Python >= 3.10
- Docker running locally, or an Oracle AI Database reachable over the network
- OCI Generative AI access to `xai.grok-3-fast`
- Local OCI config file at `~/.oci/config`, or set `OCI_CONFIG_FILE` and `OCI_PROFILE`


## 1. Install the library

Install `oracleagentmemory` from PyPI plus the OCI SDK. This OCI variant uses native OCI calls instead of provider adapters.


In [ ]:
%pip install --upgrade pip
%pip install oracleagentmemory==26.4.0 oci nest_asyncio


In [ ]:
import oracleagentmemory

## 2. Configure OCI Generative AI

The package uses an embedding model to turn text into vectors it can store and search. This guide uses `cohere.embed-english-v3.0` through the OCI SDK and `xai.grok-3-fast` for generation. Credentials are read from local OCI config; no secrets are hardcoded in the notebook.


In [ ]:
import configparser
import os
from pathlib import Path


def _expand_path(value: str) -> str:
    return os.path.expanduser(os.path.expandvars(value))


def _read_oci_profile_region(config_file: str, profile_name: str) -> str:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return ""

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT" and parser.defaults().get("region"):
        return parser.defaults()["region"].strip()
    for section in (profile_name, f"profile {profile_name}"):
        if parser.has_section(section) and parser.has_option(section, "region"):
            return parser.get(section, "region").strip()
    return ""


def _read_oci_profile_values(config_file: str, profile_name: str) -> dict[str, str]:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return {}

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT":
        values = dict(parser.defaults())
    else:
        values = {}
        for section in (profile_name, f"profile {profile_name}"):
            if parser.has_section(section):
                values.update(dict(parser.items(section)))
                break

    if values.get("key_file"):
        values["key_file"] = _expand_path(values["key_file"])
    return values


def _set_oci_env_aliases(config_file: str, profile_name: str) -> None:
    values = _read_oci_profile_values(config_file, profile_name)
    aliases = {
        "user": ("OCI_USER", "OCI_USER_ID"),
        "tenancy": ("OCI_TENANCY", "OCI_TENANCY_ID"),
        "fingerprint": ("OCI_FINGERPRINT",),
        "key_file": ("OCI_KEY_FILE",),
        "region": ("OCI_REGION", "OCI_REGION_NAME"),
    }
    for config_key, env_names in aliases.items():
        value = values.get(config_key, "")
        if not value:
            continue
        for env_name in env_names:
            os.environ.setdefault(env_name, value)


def _env(name: str, default: str = "") -> str:
    raw_value = os.environ.get(name)
    value = raw_value if raw_value not in (None, "") else default
    value = _expand_path(value) if name.endswith("_FILE") else value
    os.environ[name] = value
    return value


def _require_non_empty(value: str, name: str, hint: str) -> None:
    if not value:
        raise RuntimeError(f"Missing {name}. {hint}")


def _require_existing_file(value: str, name: str, hint: str) -> None:
    _require_non_empty(value, name, hint)
    if not Path(value).expanduser().exists():
        raise RuntimeError(f"{name} does not exist at {value}. {hint}")


LLM_PROVIDER = _env("LLM_PROVIDER", "oci_genai_native").strip().lower().replace("-", "_")
if LLM_PROVIDER not in {"oci_genai_native", "oci_native"}:
    raise RuntimeError("This OCI notebook supports LLM_PROVIDER=oci_genai_native only.")

LLM_MODEL = _env("LLM_MODEL", "xai.grok-3-fast").strip()
OCI_CONFIG_FILE = _env("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
OCI_PROFILE_REGION = _read_oci_profile_region(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_REGION = os.environ.get("OCI_GENAI_REGION", "").strip() or "us-chicago-1"
os.environ["OCI_GENAI_REGION"] = OCI_GENAI_REGION
OCI_GENAI_ENDPOINT = _env(
    "OCI_GENAI_ENDPOINT",
    f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
OCI_PROFILE_VALUES = _read_oci_profile_values(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_COMPARTMENT_OCID = _env(
    "OCI_GENAI_COMPARTMENT_OCID",
    OCI_PROFILE_VALUES.get("compartment_id") or OCI_PROFILE_VALUES.get("tenancy", ""),
).strip()
OCI_GENAI_MODEL_OCID = _env("OCI_GENAI_MODEL_OCID", LLM_MODEL).strip()
OCI_GENAI_EMBED_MODEL = _env("OCI_GENAI_EMBED_MODEL", "cohere.embed-english-v3.0")

_set_oci_env_aliases(OCI_CONFIG_FILE, OCI_PROFILE)
os.environ.setdefault("OCI_REGION", OCI_GENAI_REGION)
os.environ.setdefault("OCI_REGION_NAME", OCI_GENAI_REGION)
os.environ.setdefault("OCI_COMPARTMENT_ID", OCI_GENAI_COMPARTMENT_OCID)

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

_require_existing_file(
    OCI_CONFIG_FILE,
    "OCI_CONFIG_FILE",
    "Set OCI_CONFIG_FILE to your OCI config path, usually ~/.oci/config.",
)
_require_non_empty(OCI_PROFILE, "OCI_PROFILE", "Set OCI_PROFILE to a profile in OCI_CONFIG_FILE.")
_require_non_empty(OCI_GENAI_REGION, "OCI_GENAI_REGION", "Set OCI_GENAI_REGION for OCI Generative AI.")
_require_non_empty(
    OCI_GENAI_COMPARTMENT_OCID,
    "OCI_GENAI_COMPARTMENT_OCID",
    "Set this to a compartment OCID with OCI Generative AI permissions. The notebook falls back to compartment_id or tenancy from OCI_CONFIG_FILE when present.",
)
_require_non_empty(
    OCI_GENAI_MODEL_OCID,
    "OCI_GENAI_MODEL_OCID",
    "Set this to the on-demand model OCID or model ID. Defaults to LLM_MODEL, for example xai.grok-3-fast.",
)

import nest_asyncio
nest_asyncio.apply()

print("Environment configured for OCI Generative AI.")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {LLM_MODEL}")
print(f"OCI profile region: {OCI_PROFILE_REGION or 'not set'}")
print(f"OCI GenAI region: {OCI_GENAI_REGION}")
print(f"OCI endpoint: {OCI_GENAI_ENDPOINT}")
print(f"OCI compartment: {OCI_GENAI_COMPARTMENT_OCID[:18]}..." if OCI_GENAI_COMPARTMENT_OCID else "OCI compartment: not set")
print(f"OCI model id: {OCI_GENAI_MODEL_OCID}")
print(f"OCI embedding model: {OCI_GENAI_EMBED_MODEL}")


import asyncio
import json
from collections.abc import Sequence
from typing import Any

import numpy as np
import oci
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oracleagentmemory.apis.llms.llm import ILlm, LlmResponse


class OciSdkEmbedder(IEmbedder):
    """OracleAgentMemory embedder backed directly by OCI Generative AI."""

    def __init__(self, client, compartment_id: str, model_id: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        from oci.generative_ai_inference import models

        details = models.EmbedTextDetails()
        details.inputs = texts
        details.compartment_id = self.compartment_id
        details.serving_mode = models.OnDemandServingMode(model_id=self.model_id)
        details.truncate = models.EmbedTextDetails.TRUNCATE_END
        details.input_type = (
            models.EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY
            if is_query
            else models.EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT
        )

        response = self.client.embed_text(details)
        embeddings = getattr(response.data, "embeddings", None)
        if not embeddings:
            raise RuntimeError("OCI embed_text returned no embeddings.")
        return np.asarray(embeddings, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


def create_oci_genai_client():
    try:
        oci_config = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load OCI config profile {OCI_PROFILE!r} from {OCI_CONFIG_FILE}: {exc}"
        ) from exc

    return oci.generative_ai_inference.GenerativeAiInferenceClient(
        config=oci_config,
        service_endpoint=OCI_GENAI_ENDPOINT,
        retry_strategy=oci.retry.NoneRetryStrategy(),
        timeout=(10, 240),
    )


def _message_content_to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(getattr(item, "text", "") or getattr(item, "content", "") or item))
        return "\n".join(part for part in parts if part)
    return str(content)


def _oci_text_content(models, text: Any):
    item = models.TextContent()
    if hasattr(models.TextContent, "TYPE_TEXT"):
        item.type = models.TextContent.TYPE_TEXT
    item.text = _message_content_to_text(text)
    return item


def _chat_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))]
    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER
    oci_message.content = content_items
    return oci_message


def _extract_oci_text(message: Any) -> str:
    parts = []
    for item in getattr(message, "content", None) or []:
        text = getattr(item, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts)


def oci_chat_text(
    messages: list[dict[str, str]],
    *,
    temperature: float = 0.2,
    max_tokens: int = 1200,
    response_json_schema: dict[str, Any] | None = None,
) -> str:
    from oci.generative_ai_inference import models

    chat_request = models.GenericChatRequest()
    chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
    chat_request.messages = [_chat_message_to_oci(models, message) for message in messages]
    chat_request.max_tokens = max_tokens
    chat_request.temperature = temperature
    chat_request.top_p = 1.0
    chat_request.top_k = 0
    chat_request.is_stream = False

    if response_json_schema:
        schema = models.ResponseJsonSchema()
        schema.name = "response"
        schema.schema = response_json_schema
        schema.is_strict = False
        response_format = models.JsonSchemaResponseFormat()
        response_format.type = models.JsonSchemaResponseFormat.TYPE_JSON_SCHEMA
        response_format.json_schema = schema
        chat_request.response_format = response_format

    serving_mode = models.OnDemandServingMode()
    serving_mode.model_id = OCI_GENAI_MODEL_OCID

    chat_detail = models.ChatDetails()
    chat_detail.serving_mode = serving_mode
    chat_detail.chat_request = chat_request
    chat_detail.compartment_id = OCI_GENAI_COMPARTMENT_OCID

    response = genai_client.chat(chat_detail)
    data = response.data
    chat_response = getattr(data, "chat_response", data)
    choices = getattr(chat_response, "choices", None) or []
    message = getattr(choices[0], "message", None) if choices else None
    return _extract_oci_text(message)


class OciSdkLlm(ILlm):
    """OracleAgentMemory LLM backed directly by OCI Generative AI."""

    def __init__(self, temperature: float = 0.2, max_tokens: int = 1200):
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        if isinstance(prompt, str):
            messages = [{"role": "user", "content": prompt}]
        else:
            messages = list(prompt)
        text = oci_chat_text(
            messages,
            temperature=kwargs.get("temperature", self.temperature),
            max_tokens=kwargs.get("max_tokens", self.max_tokens),
            response_json_schema=response_json_schema,
        )
        return LlmResponse(text=text)

    async def generate_async(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        return await asyncio.to_thread(
            self.generate,
            prompt,
            response_json_schema=response_json_schema,
            **kwargs,
        )


genai_client = create_oci_genai_client()
oamp_embedder = OciSdkEmbedder(
    genai_client,
    compartment_id=OCI_GENAI_COMPARTMENT_OCID,
    model_id=OCI_GENAI_EMBED_MODEL,
)
oamp_llm = OciSdkLlm(temperature=0.2)
print("OCI Generative AI helpers ready.")


## 3. Start an Oracle AI Database instance

The package stores everything (profiles, memories, messages, embeddings) in an Oracle AI Database. The easiest way to get one running locally is via Docker:

```bash
docker run -d \
  --name oracle-free \
  -p 1521:1521 \
  -e ORACLE_PASSWORD=VectorPwd_2025 \
  gvenzl/oracle-free:23-slim
```

Wait until `docker logs oracle-free` prints `DATABASE IS READY TO USE!`, then create a dedicated `VECTOR` schema:

```bash
docker exec oracle-free sqlplus sys/VectorPwd_2025@FREEPDB1 as sysdba <<'SQL'
CREATE USER VECTOR IDENTIFIED BY VectorPwd_2025;
GRANT CONNECT, RESOURCE, UNLIMITED TABLESPACE TO VECTOR;
SQL
```

If you already have a production Oracle instance, point `DB_CONNECT_STRING` at it instead and skip the Docker step.

## 4. Connect to Oracle

The package takes a raw `oracledb.Connection`. That's it — no custom connection pool, no ORM. You own the connection lifecycle, which means you can share it with other Oracle code in the same application.

In [ ]:
import oracledb

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print(f"Connected to Oracle {connection.version}")


## 5. Create a memory client

`OracleAgentMemory` is the top-level object you'll interact with. It owns the schema, the embedder, and optionally an LLM for automatic extraction.

Key constructor arguments:

| Argument | Purpose |
|---|---|
| `connection` | Your `oracledb` connection |
| `embedder` | An embedder implementation; this OCI variant uses `OciSdkEmbedder` |
| `llm` | Optional; attaching one enables automatic memory extraction from threads |
| `extract_memories` | Whether to run extraction when messages are added to threads |
| `schema_policy` | `create_if_necessary` (safe default), `recreate` (drops and rebuilds), or `require_existing` |

For now we'll skip the LLM; we'll add one in step 13 to see automatic extraction.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory

client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    extract_memories=False,
    schema_policy="create_if_necessary",
)
print("Memory client ready.")


## 6. Register users and agents

Every memory in the package is **scoped** to a user, an agent, or both. Before you store memories, register the entities they belong to.

- `add_user(user_id, information)` — register a human user (the person the memories are *about*)
- `add_agent(agent_id, information)` — register an AI agent (the assistant *observing* and *creating* memories)

The `information` string is free-form profile text that can itself be searched.

In [ ]:
USER_ID = "dev-guide-user"
AGENT_ID = "dev-guide-agent"

for fn, eid, info in [
    (client.add_user,  USER_ID,  "Richmond — AI developer learning the AI Agent Package."),
    (client.add_agent, AGENT_ID, "Tutorial assistant for the AI Agent Package developer guide."),
]:
    try:
        fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError as exc:
        if "already exists" in str(exc):
            print(f"(already exists: {eid})")
        else:
            raise

## 7. Store your first memory

`add_memory(content, user_id=..., agent_id=..., thread_id=...)` saves a durable fact. The package embeds the content and stores both the text and its vector. Memories can be scoped to a user, an agent, a thread, or any combination.

In [ ]:
mem_id = client.add_memory(
    "Richmond is evaluating the AI Agent Package for a production RAG pipeline with 50k daily queries.",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)
print(f"Stored memory: {mem_id}")

## 8. Search memories with vector retrieval

`search(query, user_id=..., agent_id=..., max_results=...)` runs a vector similarity search over stored memories. Results come back ranked by distance (lower = more similar).

Each result carries:

- `content` — the stored text
- `distance` — how close the query embedding was
- `record` — full database record, including `record_type` (`memory`, `message`, `user`, `agent`, etc.)
- `metadata`, `timestamp`, `id`


In [ ]:
results = client.search(
    "what is the user building?",
    user_id=USER_ID,
    max_results=5,
)
for r in results:
    print(f"[{r.record.record_type}] distance={r.distance:.3f}")
    print(f"  {r.content}")


## 9. Threads: the conversation primitive

A **thread** represents one conversation. Messages you add to a thread become searchable alongside memories, and — when you attach an LLM — they are automatically summarised and mined for new memories.

Create a thread scoped to a user and agent:


In [ ]:
thread = client.create_thread(user_id=USER_ID, agent_id=AGENT_ID)
print(f"Thread ID: {thread.thread_id}")

## 10. Add messages and read them back

Use `thread.add_messages([...])` with `Message(role=..., content=...)` objects. The roles follow the usual `system` / `user` / `assistant` convention.


In [ ]:
from oracleagentmemory.apis.thread import Message

thread.add_messages([
    Message(role="user",      content="I'm building a customer support agent for a SaaS product."),
    Message(role="assistant", content="Got it. Are you focused on tier-1 routing or full issue resolution?"),
    Message(role="user",      content="Full resolution — I want the agent to look up past tickets and draft replies."),
])

for m in thread.get_messages():
    print(f"[{m.role:10s}] {m.content}")

## 11. Retrieve a context card

`thread.get_context_card()` returns an XML-formatted block of the most relevant memories for the current state of the thread. It's the single most useful method for building prompts — instead of stuffing the whole history into every request, you pull a compact, query-relevant card.


In [ ]:
def context_card_content(card, empty: str = "(no context yet)") -> str:
    """Render either the current OracleContextCard object or older string/list returns."""
    if not card:
        return empty
    if isinstance(card, str):
        return card
    if isinstance(card, (list, tuple)):
        card = card[0] if card else None
        if not card:
            return empty
    for attr in ("content", "text", "card", "xml"):
        value = getattr(card, attr, None)
        if value:
            return value
    return str(card)


card = thread.get_context_card()
print(context_card_content(card, "(no memories retrieved yet)"))


## 12. Compress the thread with `get_summary`

`thread.get_summary()` asks the LLM (if one is attached) for a compressed version of the conversation. Useful for long threads where you want to keep token counts low without losing continuity.

Without an LLM attached, it returns a plain concatenation — so let's add an LLM in the next step to see the real feature.


In [ ]:
def summary_content(summary, empty: str = "(no summary yet)") -> str:
    """Render either the current OracleSummary object or older list-style return values."""
    if not summary:
        return empty
    if isinstance(summary, (list, tuple)):
        summary = summary[0] if summary else None
        if not summary:
            return empty
    for attr in ("content", "text", "summary"):
        value = getattr(summary, attr, None)
        if value:
            return value
    return str(summary)


summary = thread.get_summary()
print(summary_content(summary, "(no summary; attach an LLM next)"))


## 13. Turn on automatic memory extraction

This is the headline feature. When you attach an `ILlm` implementation and pass `extract_memories=True`, the package will:

1. Watch every message you add to a thread
2. Periodically call the LLM to extract durable facts from recent messages
3. Store those facts as `memory` records, automatically scoped to the thread's user and agent
4. Maintain a rolling **context summary** for the thread

This OCI variant attaches `OciSdkLlm`, which calls OCI Generative AI through the native OCI SDK.


In [ ]:
llm = oamp_llm

smart_client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    llm=llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
)

smart_thread = smart_client.create_thread(
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_extraction_frequency=2,
    memory_extraction_window=4,
    enable_context_summary=True,
    context_summary_update_frequency=2,
)
print(f"Smart thread: {smart_thread.thread_id}")


## 14. Watch extraction happen

Add a few messages, then search for `record_type="memory"` — you should find facts the package extracted on its own, without you ever calling `add_memory`.

In [ ]:
smart_thread.add_messages([
    Message(role="user",      content="I prefer Python over TypeScript for backend work."),
    Message(role="assistant", content="Noted. Are you using FastAPI or something else?"),
    Message(role="user",      content="FastAPI, deployed on Oracle Cloud Infrastructure behind an API gateway."),
    Message(role="assistant", content="Makes sense for your scale. Want me to remember that stack?"),
])

extracted = smart_client.search(
    "python stack preferences",
    user_id=USER_ID,
    record_types=["memory"],
    max_results=5,
)
print(f"Extracted {len(extracted)} memories:")
for r in extracted:
    print(f"  - {r.content}")


## 15. Inspect the running summary

With `enable_context_summary=True`, `get_summary()` now produces an LLM-written compression rather than a plain concatenation.


In [ ]:
summary = smart_thread.get_summary()
print(summary_content(summary, "(no summary yet; add more messages)"))


## 16. Scope searches across users and agents

In production, your package store will have many users and agents sharing one database. `search()` takes scoping arguments so memories never leak across boundaries:

- `user_id=...` — restrict to memories that belong to this user
- `agent_id=...` — restrict to memories created by this agent
- `thread_id=...` — restrict to memories from a specific conversation
- `record_types=[...]` — filter by record kind (`memory`, `message`, `user`, `agent`, `guideline`, `fact`)
- `max_results=N` — cap the result count

Combine them freely.

In [ ]:
# Only memories, scoped to this user, this agent, this thread
scoped = smart_client.search(
    "deployment",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=smart_thread.thread_id,
    record_types=["memory"],
    max_results=3,
)
for r in scoped:
    print(f"{r.distance:.3f}  {r.content}")


## 17. Use the AI Agent Package inside an LLM agent loop

The typical pattern: **before** every LLM call, pull a context card. **After** the LLM responds, append the assistant message back to the thread. Everything else — extraction, summarisation, embedding — is automatic.

Below is a minimal agent loop using the OCI chat helper defined earlier. The same pattern works with any framework.


In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant. Use the retrieved memory to stay consistent with the user's preferences and history."


def chat_turn(user_message: str) -> str:
    smart_thread.add_messages([Message(role="user", content=user_message)])
    card = context_card_content(smart_thread.get_context_card(), "(no prior memory)")

    answer = oci_chat_text(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Memory context:\n{card}\n\nUser: {user_message}"},
        ],
        temperature=0.2,
        max_tokens=800,
    )

    smart_thread.add_messages([Message(role="assistant", content=answer)])
    return answer


print(chat_turn("What stack do I use for my backend work again?"))


## 18. Persist and resume across sessions

Everything the package stores lives in Oracle, not in your Python process. To resume a conversation from a new process, a new machine, or a new day, just reconnect and call `get_thread(thread_id)`.

In [ ]:
saved_thread_id = smart_thread.thread_id

# Imagine your program restarts here: different connection, same OCI-backed helpers.
new_connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
new_client = OracleAgentMemory(
    connection=new_connection,
    embedder=oamp_embedder,
    llm=oamp_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
)

recovered = new_client.get_thread(saved_thread_id)
print(f"Recovered thread: {recovered.thread_id}")
print(f"Messages recovered: {len(recovered.get_messages())}")

recovered.add_messages([Message(role="user", content="Are my preferences still remembered?")])
print(context_card_content(recovered.get_context_card(), "(no context yet)")[:400])


## 19. Cleanup

Delete the threads and memory you created in this walkthrough, then close both connections. The package never deletes anything on its own — you stay in control.

In [ ]:
client.delete_memory(mem_id)
smart_client.delete_thread(smart_thread.thread_id)
new_client.delete_thread(saved_thread_id) if recovered else None
connection.close()
new_connection.close()
print("Resources cleaned up.")


## Where to go next

- **`oracle_agent_memory_benchmarks_oci.ipynb`** — head-to-head comparison of an agent backed by the AI Agent Package vs a naive flat-history agent across token consumption, wall-clock latency, and response quality using OCI Generative AI.
- **`01_deep_research_oci_agents.ipynb`** — a tool-using research agent with Tavily search, OCI generation, and Oracle AI Agent Memory continuity.

**Suggested reading order for new OCI projects**
1. This guide — to understand the primitives
2. The deep-research OCI notebook — to see a tool-using agent pattern
3. The benchmarks OCI notebook — to justify the cost/quality tradeoffs to your team
